In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# GEMINI TOP-5 IMAGE LABELING
# ============================================================

import os
import re
import json
import time
import mimetypes
import pandas as pd
from tqdm import tqdm

from google import genai
from google.genai import types


# ============================================================
# CONFIG
# ============================================================

IMAGE_DIR = r"google_test_images/SMER_ViT"

INPUT_CSV = r"data/Labels_Top5.csv"

OUTPUT_CSV = r"data/Labels_Top5_Gemini_Adv_SMER.csv"

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
MODEL = "gemini-3.6-flash"

MAX_RETRIES = 5
SLEEP_BETWEEN_REQUESTS = 0.3


# ============================================================
# CLIENT
# ============================================================

client = genai.Client(
    api_key=GEMINI_API_KEY
)


# ============================================================
# PROMPT
# ============================================================

PROMPT = """
Examine this image carefully.

Identify the main object shown in the image.

Give exactly five most likely object categories, ordered from
most likely to least likely.

Rules:
- Give exactly 5 guesses.
- Use short common object/category names.
- Do not explain your reasoning.
- Do not add any text outside the JSON.
- Do not use numbering.
- Return valid JSON exactly in this format:

{
  "top5": [
    "label1",
    "label2",
    "label3",
    "label4",
    "label5"
  ]
}
"""


# ============================================================
# PARSE OUTPUT
# ============================================================

def parse_top5(text):

    if text is None:
        return None

    text = text.strip()

    # Remove markdown code fences
    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^```\s*",
        "",
        text
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    # First try normal JSON
    try:

        data = json.loads(text)

        labels = data.get("top5", [])

        if isinstance(labels, list) and len(labels) >= 5:

            return [
                str(x).strip()
                for x in labels[:5]
            ]

    except Exception:
        pass


    # Fallback: extract quoted strings
    quoted = re.findall(
        r'"([^"]+)"',
        text
    )

    quoted = [
        x for x in quoted
        if x.lower() != "top5"
    ]

    if len(quoted) >= 5:
        return quoted[:5]

    return None


# ============================================================
# QUERY GEMINI
# ============================================================

def query_gemini(image_path):

    # Detect MIME type
    mime_type, _ = mimetypes.guess_type(
        image_path
    )

    if mime_type is None:
        mime_type = "image/png"


    # Read image bytes
    with open(image_path, "rb") as f:
        image_bytes = f.read()


    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type
    )


    response = client.models.generate_content(

        model=MODEL,

        contents=[
            image_part,
            PROMPT
        ],

        config=types.GenerateContentConfig(

            temperature=0.0,

            response_mime_type="application/json"
        )
    )


    text = response.text

    labels = parse_top5(text)

    return labels, text


# ============================================================
# LOAD CSV
# ============================================================

if os.path.exists(OUTPUT_CSV):

    print("Existing output found.")
    print("Resuming:", OUTPUT_CSV)

    df = pd.read_csv(OUTPUT_CSV)

else:

    df = pd.read_csv(INPUT_CSV)


# ============================================================
# REQUIRED COLUMNS
# ============================================================

label_columns = [

    "Label 1",
    "Label 2",
    "Label 3",
    "Label 4",
    "Label 5"
]

for col in label_columns:

    if col not in df.columns:
        df[col] = ""


if "status" not in df.columns:
    df["status"] = ""

if "raw_response" not in df.columns:
    df["raw_response"] = ""


print()
print("Total images:", len(df))
print("Image folder:", IMAGE_DIR)
print("Model:", MODEL)
print()


# ============================================================
# MAIN LOOP
# ============================================================

for idx in tqdm(
    range(len(df)),
    desc="Gemini labeling"
):

    image_name = str(
        df.loc[idx, "image_name"]
    )

    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )


    # --------------------------------------------------------
    # SKIP COMPLETED
    # --------------------------------------------------------

    already_done = all(

        pd.notna(df.loc[idx, col])

        and

        str(
            df.loc[idx, col]
        ).strip() != ""

        for col in label_columns
    )

    if already_done:
        continue


    # --------------------------------------------------------
    # IMAGE NOT FOUND
    # --------------------------------------------------------

    if not os.path.exists(image_path):

        print(
            "\nImage not found:",
            image_path
        )

        df.loc[
            idx,
            "status"
        ] = "image_not_found"

        df.to_csv(
            OUTPUT_CSV,
            index=False
        )

        continue


    # --------------------------------------------------------
    # RETRIES
    # --------------------------------------------------------

    success = False

    for attempt in range(MAX_RETRIES):

        try:

            labels, raw_response = query_gemini(
                image_path
            )


            df.loc[
                idx,
                "raw_response"
            ] = raw_response


            if labels is None or len(labels) != 5:

                raise ValueError(
                    f"Could not parse labels: {raw_response}"
                )


            # ------------------------------------------------
            # SAVE LABELS
            # ------------------------------------------------

            for j in range(5):

                df.loc[
                    idx,
                    f"Label {j+1}"
                ] = labels[j]


            df.loc[
                idx,
                "status"
            ] = "success"

            success = True

            break


        except Exception as e:

            print(
                f"\nError on {image_name}"
            )

            print(
                f"Attempt {attempt+1}/{MAX_RETRIES}"
            )

            print(e)

            df.loc[
                idx,
                "status"
            ] = f"failed_attempt_{attempt+1}"

            time.sleep(
                2 ** attempt
            )


    # --------------------------------------------------------
    # SAVE AFTER EVERY IMAGE
    # --------------------------------------------------------

    df.to_csv(
        OUTPUT_CSV,
        index=False
    )


    if not success:

        print(
            "FAILED:",
            image_name
        )


    time.sleep(
        SLEEP_BETWEEN_REQUESTS
    )


# ============================================================
# FINAL SAVE
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

success_count = (
    df["status"] == "success"
).sum()

failed_count = (
    df["status"] != "success"
).sum()


print()
print("===================================")
print("FINISHED")
print("===================================")

print("Total:", len(df))

print(
    "Successful:",
    success_count
)

print(
    "Failed / missing:",
    failed_count
)

print()
print(
    "Saved:",
    OUTPUT_CSV
)

In [ ]:
# ============================================================
# RETRY ONLY FAILED / INCOMPLETE GEMINI ROWS
# ============================================================

import os
import time
import pandas as pd
from tqdm import tqdm


OUTPUT_CSV = r"data/Labels_Top5_Gemini_Adv_SMER.csv"

label_columns = [
    "Label 1",
    "Label 2",
    "Label 3",
    "Label 4",
    "Label 5"
]


# ============================================================
# LOAD EXISTING RESULTS
# ============================================================

df = pd.read_csv(OUTPUT_CSV)

print("Total rows:", len(df))


# ============================================================
# FIND INCOMPLETE ROWS
# ============================================================

def row_is_complete(row):
    return all(
        pd.notna(row[col])
        and str(row[col]).strip() != ""
        for col in label_columns
    )


incomplete_indices = [
    idx
    for idx, row in df.iterrows()
    if not row_is_complete(row)
]

print("Incomplete rows:", len(incomplete_indices))


# ============================================================
# RETRY
# ============================================================

MAX_RETRIES = 8

for idx in tqdm(
    incomplete_indices,
    desc="Retrying failed Gemini images"
):

    image_name = str(
        df.loc[idx, "image_name"]
    )

    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )

    if not os.path.exists(image_path):

        print("Missing image:", image_path)

        df.loc[
            idx,
            "status"
        ] = "image_not_found"

        continue


    success = False


    for attempt in range(MAX_RETRIES):

        try:

            labels, raw_response = query_gemini(
                image_path
            )


            if labels is None or len(labels) != 5:

                raise ValueError(
                    f"Could not parse labels: {raw_response}"
                )


            # -----------------------------------------------
            # WRITE LABELS
            # -----------------------------------------------

            for j in range(5):

                df.loc[
                    idx,
                    f"Label {j+1}"
                ] = labels[j]


            df.loc[
                idx,
                "raw_response"
            ] = raw_response


            df.loc[
                idx,
                "status"
            ] = "success"


            success = True

            print(
                f"Recovered: {image_name}"
            )

            break


        except Exception as e:

            print(
                f"\nRetry error: {image_name}"
            )

            print(
                f"Attempt {attempt+1}/{MAX_RETRIES}"
            )

            print(e)


            # Longer backoff for Gemini overload
            delay = min(
                60,
                2 ** attempt
            )

            print(
                f"Waiting {delay}s..."
            )

            time.sleep(delay)


    # ========================================================
    # SAVE AFTER EACH IMAGE
    # ========================================================

    df.to_csv(
        OUTPUT_CSV,
        index=False
    )


    if not success:

        df.loc[
            idx,
            "status"
        ] = "still_failed"

        print(
            "Still failed:",
            image_name
        )


# ============================================================
# FINAL SAVE
# ============================================================

df.to_csv(
    OUTPUT_CSV,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

complete_count = sum(
    row_is_complete(row)
    for _, row in df.iterrows()
)

remaining_count = (
    len(df) - complete_count
)


print("\n====================================")
print("RETRY FINISHED")
print("====================================")

print("Total:", len(df))
print("Complete:", complete_count)
print("Still incomplete:", remaining_count)

print(
    "Saved:",
    OUTPUT_CSV
)